In [61]:
import xml.etree.ElementTree as ET
import pandas as pd

# Load the XML file
capec_file = "capec_v3.9.xml"
tree = ET.parse(capec_file)
root = tree.getroot()

# Define the XML namespace
namespace = {
    "capec": "http://capec.mitre.org/capec-3",
    "xhtml": "http://www.w3.org/1999/xhtml"
}

# List to store extracted data
data = []

# Iterate through Attack Patterns
for attack_pattern in root.findall(".//capec:Attack_Patterns/capec:Attack_Pattern", namespace):
    capec_id = attack_pattern.get("ID", "N/A")
    name = attack_pattern.get("Name", "N/A")
    abstraction = attack_pattern.get("Abstraction", "N/A")
    status = attack_pattern.get("Status", "N/A")

    # Extract Description (Handles XHTML Tags)
    description = attack_pattern.find("capec:Description", namespace)
    capec_desc_text = " ".join(description.itertext()).strip() if description is not None else "N/A"

    # Extract Extended Description (Handles Direct Text & XHTML Nested Elements)
    extended_description = attack_pattern.find("capec:Extended_Description", namespace)
    ext_desc_text = " ".join(extended_description.itertext()).strip() if extended_description is not None else "N/A"

    # Extract Likelihood of Attack & Typical Severity
    likelihood = attack_pattern.find("capec:Likelihood_Of_Attack", namespace)
    severity = attack_pattern.find("capec:Typical_Severity", namespace)

    likelihood_text = likelihood.text.strip() if likelihood is not None and likelihood.text else "N/A"
    severity_text = severity.text.strip() if severity is not None and severity.text else "N/A"

    # Extract Related Attack Patterns
    related_patterns = [
        f"CAPEC-{related.get('CAPEC_ID', 'N/A')} ({related.get('Nature', 'N/A')})"
        for related in attack_pattern.findall(".//capec:Related_Attack_Patterns/capec:Related_Attack_Pattern", namespace)
    ]
    related_patterns_text = ", ".join(related_patterns) if related_patterns else "N/A"

    # Extract Execution Flow (Steps, Phase, Description, Techniques)
    execution_flow = []
    for step in attack_pattern.findall(".//capec:Execution_Flow/capec:Attack_Step", namespace):
        step_number_text = step.findtext("capec:Step", default="N/A", namespaces=namespace).strip()
        phase_text = step.findtext("capec:Phase", default="N/A", namespaces=namespace).strip()
        step_desc_text = step.findtext("capec:Description", default="N/A", namespaces=namespace).strip()

        techniques = [
            tech.text.strip() for tech in step.findall("capec:Technique", namespace) if tech.text
        ]
        techniques_text = ", ".join(techniques) if techniques else "N/A"

        execution_flow.append(f"Step {step_number_text}: {phase_text} - {step_desc_text} (Techniques: {techniques_text})")

    execution_flow_text = " || ".join(execution_flow) if execution_flow else "N/A"

    # Extract Prerequisites
    prerequisites = [
        pre.text.strip() for pre in attack_pattern.findall(".//capec:Prerequisites/capec:Prerequisite", namespace) if pre.text
    ]
    prerequisites_text = " || ".join(prerequisites) if prerequisites else "N/A"

    # Extract Skills Required
    skills_required = [
        f"{skill.get('Level', 'N/A')}: {skill.text.strip() if skill.text else 'N/A'}"
        for skill in attack_pattern.findall(".//capec:Skills_Required/capec:Skill", namespace)
    ]
    skills_text = " || ".join(skills_required) if skills_required else "N/A"

    # Extract Resources Required
    resources = [
        res.text.strip() for res in attack_pattern.findall(".//capec:Resources_Required/capec:Resource", namespace) if res.text
    ]
    resources_text = " || ".join(resources) if resources else "N/A"

    # Extract Consequences (Scope & Impact)
    consequences_list = [
        f"Scope: {', '.join([scope.text.strip() for scope in consequence.findall('capec:Scope', namespace) if scope.text])} | "
        f"Impact: {', '.join([impact.text.strip() for impact in consequence.findall('capec:Impact', namespace) if impact.text])}"
        for consequence in attack_pattern.findall(".//capec:Consequences/capec:Consequence", namespace)
    ]
    consequences_text = " || ".join(consequences_list) if consequences_list else "N/A"

    # Extract Mitigations
    mitigations = [
        " ".join([p.text.strip() for p in mitigation.findall(".//{http://www.w3.org/1999/xhtml}p") if p.text])
        if mitigation.findall(".//{http://www.w3.org/1999/xhtml}p")
        else mitigation.text.strip() if mitigation.text else "N/A"
        for mitigation in attack_pattern.findall(".//capec:Mitigations/capec:Mitigation", namespace)
    ]
    mitigations_text = " || ".join(mitigations) if mitigations else "N/A"

    # Extract Example Instances
    examples = [
        " ".join([p.text.strip() for p in example.findall(".//{http://www.w3.org/1999/xhtml}p") if p.text])
        if example.findall(".//{http://www.w3.org/1999/xhtml}p")
        else example.text.strip() if example.text else "N/A"
        for example in attack_pattern.findall(".//capec:Example_Instances/capec:Example", namespace)
    ]
    examples_text = " || ".join(examples) if examples else "N/A"

    # Extract Related Weaknesses (CWE IDs)
    related_weaknesses = [
        f"CWE-{rw.get('CWE_ID', 'N/A')}" for rw in attack_pattern.findall(".//capec:Related_Weaknesses/capec:Related_Weakness", namespace)
    ]
    related_weaknesses_text = ", ".join(related_weaknesses) if related_weaknesses else "N/A"

    # Extract Taxonomy Mappings
    taxonomy_mappings = [
        f"{tm.get('Taxonomy_Name', 'N/A')}: {tm.findtext('capec:Entry_ID', default='N/A', namespaces=namespace)} - "
        f"{tm.findtext('capec:Entry_Name', default='N/A', namespaces=namespace)}"
        for tm in attack_pattern.findall(".//capec:Taxonomy_Mappings/capec:Taxonomy_Mapping", namespace)
    ]
    taxonomy_text = " || ".join(taxonomy_mappings) if taxonomy_mappings else "N/A"

    # Append extracted data
    data.append({
        "CAPEC_ID": capec_id,
        "Name": name,
        "Abstraction": abstraction,
        "Status": status,
        "Description": capec_desc_text,
        "Extended_Description": ext_desc_text,
        "Likelihood_Of_Attack": likelihood_text,
        "Typical_Severity": severity_text,
        "Related_Attack_Patterns": related_patterns_text,
        "Execution_Flow": execution_flow_text,
        "Prerequisites": prerequisites_text,
        "Skills_Required": skills_text,
        "Resources_Required": resources_text,
        "Consequences": consequences_text,
        "Mitigations": mitigations_text,
        "Example_Instances": examples_text,
        "Related_Weaknesses": related_weaknesses_text,
        "Taxonomy_Mappings": taxonomy_text,
    })

# Convert to DataFrame
df = pd.DataFrame(data)
df


,CAPEC_ID,Name,Abstraction,Status,Description,Extended_Description,Likelihood_Of_Attack,Typical_Severity,Related_Attack_Patterns,Execution_Flow,Prerequisites,Skills_Required,Resources_Required,Consequences,Mitigations,Example_Instances,Related_Weaknesses,Taxonomy_Mappings
0,1,Accessing Functionality Not Properly Constrain...,Standard,Draft,"In applications, particularly web applications...",N/A,High,High,"CAPEC-122 (ChildOf), CAPEC-17 (CanPrecede)",Step 1: Explore - [Survey] The attacker survey...,The application must be navigable in a manner ...,Low: In order to discover unrestricted resourc...,None: No specialized resources are required to...,"Scope: Confidentiality, Access Control, Author...","In a J2EE setting, administrators can associat...",Implementing the Model-View-Controller (MVC) w...,"CWE-276, CWE-285, CWE-434, CWE-693, CWE-732, C...",ATTACK: 1574.010 - Hijack Execution Flow: Serv...
1,10,Buffer Overflow via Environment Variables,Detailed,Draft,This attack pattern involves causing a buffer ...,Although the focus of this attack is putting e...,High,High,CAPEC-100 (ChildOf),Step 1: Explore - [Identify target application...,The application uses environment variables. ||...,Low: An attacker can simply overflow a buffer ...,N/A,Scope: Availability | Impact: Unreliable Execu...,Do not expose environment variable to the user...,A buffer overflow in sccw allows local users t...,"CWE-120, CWE-302, CWE-118, CWE-119, CWE-74, CW...",OWASP Attacks: N/A - Buffer Overflow via Envir...
2,100,Overflow Buffers,Standard,Draft,Buffer Overflow attacks target improper or mis...,N/A,High,Very High,CAPEC-123 (ChildOf),Step 1: Explore - [Identify target application...,Targeted software performs buffer operations. ...,"Low: In most cases, overflowing a buffer does ...",None: No specialized resources are required to...,Scope: Availability | Impact: Unreliable Execu...,Use a language or compiler that performs autom...,The most straightforward example is an applica...,"CWE-120, CWE-119, CWE-131, CWE-129, CWE-805, C...",WASC: 07 - Buffer Overflow || OWASP Attacks: N...
3,101,Server Side Include (SSI) Injection,Detailed,Draft,An attacker can use Server Side Include (SSI) ...,N/A,High,High,"CAPEC-253 (ChildOf), CAPEC-600 (CanPrecede)",Step 1: Explore - [Determine applicability] Th...,A web server that supports server side include...,Medium: The attacker needs to be aware of SSI ...,None: No specialized resources are required to...,Scope: Confidentiality | Impact: Read Data || ...,Set the OPTIONS IncludesNOEXEC in the global a...,Consider a website hosted on a server that per...,"CWE-97, CWE-74, CWE-20",WASC: 36 - SSI Injection || OWASP Attacks: N/A...
4,102,Session Sidejacking,Detailed,Draft,Session sidejacking takes advantage of an unen...,N/A,High,High,CAPEC-593 (ChildOf),Step 1: Explore - [Detect Unprotected Session ...,An attacker and the victim are both using the ...,Low: Easy to use tools exist to automate this ...,"A packet sniffing tool, such as wireshark, can...","Scope: Confidentiality, Access Control, Author...",Make sure that HTTPS is used to communicate wi...,The attacker and the victim are using the same...,"CWE-294, CWE-522, CWE-523, CWE-319, CWE-614",N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
610,95,WSDL Scanning,Detailed,Draft,This attack targets the WSDL interface made av...,N/A,High,High,CAPEC-54 (ChildOf),Step 1: Explore - [Scan for WSDL Documents] Th...,A client program connecting to a web service c...,Low: This attack can be as simple as reading W...,N/A,Scope: Confidentiality | Impact: Read Data,It is important to protect WSDL file or provid...,A WSDL interface may expose a function vulnera...,CWE-538,N/A
611,96,Block Access to Libraries,Detailed,Draft,An application typically makes calls to functi...,N/A,Medium,Medium,CAPEC-603 (ChildOf),Step 1: Explore - Determine what external libr...,An application requires access to external lib...,Low: Knowledge of how to block access to libra...,N

In [62]:
# Function to extract MITRE ATT&CK techniques (Entry_ID and Entry_Name)
def extract_mitre_techniques(taxonomy_mappings):
    if taxonomy_mappings == "N/A":
        return "N/A"
    
    mitre_techniques = [
        mapping.split(": ")[1]  # Extract "Entry_ID - Entry_Name"
        for mapping in taxonomy_mappings.split(" || ")
        if mapping.startswith("ATTACK:")  # Filter only ATT&CK mappings
    ]
    
    return " || ".join(mitre_techniques) if mitre_techniques else "N/A"

# Apply the function to create a new column
df["Mitre_Techniques"] = df["Taxonomy_Mappings"].apply(extract_mitre_techniques)

# Display the updated DataFrame
df


,CAPEC_ID,Name,Abstraction,Status,Description,Extended_Description,Likelihood_Of_Attack,Typical_Severity,Related_Attack_Patterns,Execution_Flow,Prerequisites,Skills_Required,Resources_Required,Consequences,Mitigations,Example_Instances,Related_Weaknesses,Taxonomy_Mappings,Mitre_Techniques
0,1,Accessing Functionality Not Properly Constrain...,Standard,Draft,"In applications, particularly web applications...",N/A,High,High,"CAPEC-122 (ChildOf), CAPEC-17 (CanPrecede)",Step 1: Explore - [Survey] The attacker survey...,The application must be navigable in a manner ...,Low: In order to discover unrestricted resourc...,None: No specialized resources are required to...,"Scope: Confidentiality, Access Control, Author...","In a J2EE setting, administrators can associat...",Implementing the Model-View-Controller (MVC) w...,"CWE-276, CWE-285, CWE-434, CWE-693, CWE-732, C...",ATTACK: 1574.010 - Hijack Execution Flow: Serv...,1574.010 - Hijack Execution Flow
1,10,Buffer Overflow via Environment Variables,Detailed,Draft,This attack pattern involves causing a buffer ...,Although the focus of this attack is putting e...,High,High,CAPEC-100 (ChildOf),Step 1: Explore - [Identify target application...,The application uses environment variables. ||...,Low: An attacker can simply overflow a buffer ...,N/A,Scope: Availability | Impact: Unreliable Execu...,Do not expose environment variable to the user...,A buffer overflow in sccw allows local users t...,"CWE-120, CWE-302, CWE-118, CWE-119, CWE-74, CW...",OWASP Attacks: N/A - Buffer Overflow via Envir...,N/A
2,100,Overflow Buffers,Standard,Draft,Buffer Overflow attacks target improper or mis...,N/A,High,Very High,CAPEC-123 (ChildOf),Step 1: Explore - [Identify target application...,Targeted software performs buffer operations. ...,"Low: In most cases, overflowing a buffer does ...",None: No specialized resources are required to...,Scope: Availability | Impact: Unreliable Execu...,Use a language or compiler that performs autom...,The most straightforward example is an applica...,"CWE-120, CWE-119, CWE-131, CWE-129, CWE-805, C...",WASC: 07 - Buffer Overflow || OWASP Attacks: N...,N/A
3,101,Server Side Include (SSI) Injection,Detailed,Draft,An attacker can use Server Side Include (SSI) ...,N/A,High,High,"CAPEC-253 (ChildOf), CAPEC-600 (CanPrecede)",Step 1: Explore - [Determine applicability] Th...,A web server that supports server side include...,Medium: The attacker needs to be aware of SSI ...,None: No specialized resources are required to...,Scope: Confidentiality | Impact: Read Data || ...,Set the OPTIONS IncludesNOEXEC in the global a...,Consider a website hosted on a server that per...,"CWE-97, CWE-74, CWE-20",WASC: 36 - SSI Injection || OWASP Attacks: N/A...,N/A
4,102,Session Sidejacking,Detailed,Draft,Session sidejacking takes advantage of an unen...,N/A,High,High,CAPEC-593 (ChildOf),Step 1: Explore - [Detect Unprotected Session ...,An attacker and the victim are both using the ...,Low: Easy to use tools exist to automate this ...,"A packet sniffing tool, such as wireshark, can...","Scope: Confidentiality, Access Control, Author...",Make sure that HTTPS is used to communicate wi...,The attacker and the victim are using the same...,"CWE-294, CWE-522, CWE-523, CWE-319, CWE-614",N/A,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
610,95,WSDL Scanning,Detailed,Draft,This attack targets the WSDL interface made av...,N/A,High,High,CAPEC-54 (ChildOf),Step 1: Explore - [Scan for WSDL Documents] Th...,A client program connecting to a web service c...,Low: This attack can be as simple as reading W...,N/A,Scope: Confidentiality | Impact: Read Data,It is important to protect WSDL file or provid...,A WSDL interface may expose a function vulnera...,CWE-538,N/A,N/A
611,96,Block Access to Libraries,Detailed,Draft,An application typically makes calls to functi...,N/A,Medium,Medium,CAPEC-603 (ChildOf),Step 1: Explore - Determine what external libr...,An application requires acc

In [ ]:
df.to_excel("../../data_merge/Datasets/capec_data.xlsx")